In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [3]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [4]:
files

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

In [5]:
shared_path = "../Data/R3Data"

### 6. vitals
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [6]:
idx = 6
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

vitals
['PATIENT_STUDY_ID', 'DATE_TAKEN', 'WEIGHT', 'WEIGHT_UNIT', 'HEIGHT', 'HEIGHT_UNIT', 'BMI']


In [7]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['vitals.txt', 'vitals.csv', 'vitals.csv']

### Cancer

In [8]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [9]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [10]:
final_df

,PATIENT_STUDY_ID,VISIT_ID,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,DATE_TAKEN,BMI
0,4335887037,254619441,1748.8,OZ,60.5,IN,12/19/2018 00:00:00,20.99
1,4335887037,136225343,1744.0,OZ,61.5,IN,10/23/2020 00:00:00,20.26
2,4335887037,302203019,1776.0,OZ,61.5,IN,09/26/2016 00:00:00,20.63
3,4335887037,394368130,1792.0,OZ,61.5,IN,11/17/2016 00:00:00,20.82
4,4335887037,133244636,1737.6,OZ,59.5,IN,09/13/2021 00:00:00,21.57
...,...,...,...,...,...,...,...,...
262140,4333799158,284349142,2444.8,OZ,64.0,IN,07/22/2019 00:00:00,26.23
262141,4333799158,251247940,2400.0,OZ,64.0,IN,01/31/2018 00:00:00,25.75
262142,4333799158,195391960,2448.0,OZ,64.0,IN,06/18/2021 00:00:00,26.26
262143,4333799158,157324617,2448.0,OZ,64.5,IN,06/24/2020 00:00:00,25.86


In [11]:
final_df['DATE_TAKEN'] = pd.to_datetime(final_df['DATE_TAKEN'], format='%m/%d/%Y %H:%M:%S')
final_df['DATE_TAKEN'] = final_df['DATE_TAKEN'].dt.strftime('%Y-%m-%d')

In [12]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'DATE_TAKEN'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [13]:
unique_df = final_df.copy()

In [14]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [15]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [16]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [17]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,VISIT_ID,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,DATE_TAKEN,BMI,duplicate_count
0,4330018595,299190495,2208.0,OZ,63.500,IN,2019-06-24,24.06,1
1,4330018595,155957250,2176.0,OZ,63.500,IN,2020-03-03,23.71,1
2,4330018595,144772076,2000.0,OZ,63.500,IN,2020-06-23,21.80,1
3,4330018595,142308452,2000.0,OZ,63.500,IN,2020-07-07,21.80,1
4,4330018595,141896519,2000.0,OZ,63.500,IN,2020-08-11,21.80,1
5,4330018595,139133438,2048.0,OZ,63.500,IN,2020-10-23,22.32,1
6,4330018595,125676010,2016.0,OZ,63.500,IN,2020-12-02,21.97,1
7,4330018595,120530956,2032.0,OZ,66.000,IN,2021-01-22,20.50,1
8,4330018595,146859701,2080.0,OZ,62.500,IN,2021-08-11,23.40,1
9,4330018595,3955452932415,2066.0,OZ,NaN,IN,2021-10-20,23.24,1


In [18]:
unique_df['duplicate_count'].unique()

array([1, 2, 3, 5, 4, 6, 8, 7])

In [19]:
unique_df[extract_cols]

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI
0,4330018595,2019-06-24,2208.0,OZ,63.5,IN,24.06
1,4330018595,2020-03-03,2176.0,OZ,63.5,IN,23.71
2,4330018595,2020-06-23,2000.0,OZ,63.5,IN,21.80
3,4330018595,2020-07-07,2000.0,OZ,63.5,IN,21.80
4,4330018595,2020-08-11,2000.0,OZ,63.5,IN,21.80
...,...,...,...,...,...,...,...
251364,4339959661,2021-03-02,2896.0,OZ,64.0,IN,31.07
251365,4339959661,2021-04-02,2832.0,OZ,64.0,IN,30.38
251366,4339959661,2021-04-12,2912.0,OZ,64.0,IN,31.24
251367,4339959661,2021-04-14,2880.0,OZ,64.0,IN,30.90


In [20]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)

### Control

In [21]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['vitals_controls.txt', 'vitals_controls.csv']

In [22]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [23]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [24]:
final_df

,PATIENT_STUDY_ID,VISIT_ID,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,DATE_TAKEN,BMI
0,4334386407,371324658,2534.4,OZ,NaN,IN,07/05/2017 00:00:00,26.36
1,4333435902,254594103,2710.4,OZ,62.0,IN,12/14/2017 00:00:00,30.98
2,4334027147,5162225098387,NaN,KG,NaN,CM,10/25/2017 14:03:00,25.70
3,4334048893,5033184548315,NaN,KG,NaN,CM,08/31/2017 13:52:00,28.70
4,4333392107,5387148648467,71.0,KG,NaN,CM,07/25/2017 09:15:00,29.60
...,...,...,...,...,...,...,...,...
892321,4334314069,5560254053309,NaN,KG,NaN,CM,12/04/2022 09:27:00,29.20
892322,4334037160,5054773913220,NaN,KG,NaN,CM,12/03/2022 18:12:00,29.20
892323,4333236698,5399021033223,NaN,KG,NaN,CM,12/09/2022 11:44:00,41.50
892324,4334135035,5561899953217,NaN,KG,NaN,CM,12/16/2022 12:43:00,17.60


In [25]:
final_df['DATE_TAKEN'] = pd.to_datetime(final_df['DATE_TAKEN'], format='%m/%d/%Y %H:%M:%S')
final_df['DATE_TAKEN'] = final_df['DATE_TAKEN'].dt.strftime('%Y-%m-%d')

In [26]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'DATE_TAKEN'], ascending=[True, True], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [27]:
unique_df = final_df.copy()

In [28]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [29]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [30]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [31]:
unique_df['duplicate_count'].unique()

array([1, 2, 4, 3, 5, 6, 7, 8])

In [32]:
unique_df[unique_extract_cols]

,PATIENT_STUDY_ID,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,duplicate_count
0,4330000534,2017-08-30,2480.00,OZ,65.0,IN,25.79,1
1,4330000534,2018-01-03,2480.00,OZ,65.0,IN,25.79,1
2,4330000534,2018-03-14,2476.80,OZ,65.0,IN,25.76,1
3,4330000534,2018-04-04,2222.24,OZ,65.0,IN,23.11,1
4,4330000534,2018-04-13,2433.88,OZ,65.0,IN,25.31,1
...,...,...,...,...,...,...,...,...
860766,4339982047,2022-07-11,2800.00,OZ,64.0,IN,30.04,1
860767,4339982047,2022-07-27,79.00,KG,NaN,CM,30.10,1
860768,4339988199,2022-03-04,3280.00,OZ,61.0,IN,38.73,1
860769,4339988199,2022-04-21,3216.00,OZ,61.0,IN,37.98,1


In [33]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)